# Etapa 4 — Complemento: EM e perfis latentes

Usa somente 2024; preparação independente dentro de cada fold. Modelo de mistura gaussiana diagonal por classe. **K=1** representa Gaussian Naive Bayes; **K=2 ou 3** incorpora perfis latentes aprendidos por EM.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from utils import carregar_dados, iterar_folds, preparar_particao
from em import NaiveBayesComEM

dados = carregar_dados(2024)
print("Registros 2024:", len(dados))

## Experimento 1 — Um fold espacial e a curva de convergência

Executamos primeiro um único fold, para verificar se as implementações funcionam sobre os dados reais.

In [ ]:
fold, treino, validacao = next(iterar_folds(dados, "espacial"))
p = preparar_particao(treino, validacao, "padronizada")
Xtr = p.X_treino.to_numpy(dtype=float)
Xva = p.X_avaliacao.to_numpy(dtype=float)
ytr = p.y_cls_treino.to_numpy()
yva = p.y_cls_avaliacao.to_numpy()

modelos = {}
linhas = []
for k in (1, 2, 3):
    m = NaiveBayesComEM(n_componentes=k, max_iter=100, tol=1e-6, reg_covar=1e-4, random_state=42)
    m.fit(Xtr, ytr)
    prob = m.predict_proba(Xva)[:, 1]
    pred = (prob >= 0.5).astype(int)
    modelos[k] = m
    linhas.append({
        "modelo": "Gaussian NB" if k == 1 else f"NB + EM ({k} perfis/classe)",
        "acuracia": accuracy_score(yva, pred),
        "precisao": precision_score(yva, pred, zero_division=0),
        "recall": recall_score(yva, pred, zero_division=0),
        "f1": f1_score(yva, pred, zero_division=0),
        "roc_auc": roc_auc_score(yva, prob),
        "positivos_previstos": int(pred.sum()),
    })
display(pd.DataFrame(linhas))

In [ ]:
plt.figure(figsize=(9, 5))
for k in (2, 3):
    for classe, mistura in zip(modelos[k].classes_, modelos[k].modelos_):
        historico = mistura.log_verossimilhanca_
        plt.plot(range(len(historico)), historico, marker=".", markersize=3, label=f"K={k}, classe={classe}")
        diferencas = np.diff(historico)
        print(f"K={k}, classe={classe}: {mistura.n_iter_} iterações; convergiu={mistura.convergiu_}; "
              f"menor variação={diferencas.min():.6f}")
plt.xlabel("Iteração do EM (0 = inicialização)")
plt.ylabel("Log-verossimilhança de treino")
plt.title("Convergência do EM no primeiro fold espacial")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Perfis latentes: análise descritiva

As médias abaixo estão em escala padronizada e **não** significam tipos ambientais confirmados. A interpretação exige analisar a origem de cada atributo e limitações dos dados.

In [ ]:
nomes_atributos = list(p.X_treino.columns)
for k in (2, 3):
    for classe, mistura in zip(modelos[k].classes_, modelos[k].modelos_):
        print(f"\nClasse {classe}, K={k} | Pesos dos perfis:", np.round(mistura.pesos_, 3))
        tabela = pd.DataFrame(mistura.medias_, columns=nomes_atributos)
        tabela.index = [f"perfil_{i+1}" for i in range(k)]
        display(tabela.round(3))

## Experimento 2 — Cinco folds espaciais

Execute depois de validar o experimento de um fold. Compara os três modelos nos mesmos folds e no limiar pré-fixado de 0,5; nenhuma parametrização é escolhida com base nos folds externos.

In [ ]:
resultados = []
for fold, treino, validacao in iterar_folds(dados, "espacial"):
    p = preparar_particao(treino, validacao, "padronizada")
    Xtr = p.X_treino.to_numpy(dtype=float)
    Xva = p.X_avaliacao.to_numpy(dtype=float)
    ytr = p.y_cls_treino.to_numpy()
    yva = p.y_cls_avaliacao.to_numpy()
    for k in (1, 2, 3):
        m = NaiveBayesComEM(n_componentes=k, max_iter=100, tol=1e-6, reg_covar=1e-4, random_state=42).fit(Xtr, ytr)
        prob = m.predict_proba(Xva)[:, 1]
        pred = (prob >= 0.5).astype(int)
        resultados.append({"fold": fold, "K": k, "acuracia": accuracy_score(yva, pred),
                           "precisao": precision_score(yva, pred, zero_division=0),
                           "recall": recall_score(yva, pred, zero_division=0),
                           "f1": f1_score(yva, pred, zero_division=0),
                           "roc_auc": roc_auc_score(yva, prob)})
    print(f"Fold {fold} concluído")

resultados_em = pd.DataFrame(resultados)
display(resultados_em)
display(resultados_em.groupby("K")[["acuracia", "precisao", "recall", "f1", "roc_auc"]].agg(["mean", "std"]))

## Observações metodológicas

- O método EM ajusta a verossimilhança, não diretamente a acurácia ou o F1.
- Compare *Gaussian NB (K=1)* com *NB+EM (K=2/3)* para isolar o efeito de perfis latentes.
- A classificação com Bernoulli NB da parte anterior permanece uma comparação **adicional**; sua representação de entrada é diferente.
- O uso de 2025 continua reservado à avaliação final.
- Não conclua que houve melhora antes de inspecionar as métricas de todos os folds.